# Summarization Evaluation: Entity Coverage F1

This notebook evaluates the **EntiLytics summarization feature** using **Entity Coverage F1** as the evaluation metric.

---

## Overview

EntiLytics uses an extractive summarization approach. It selects and re-orders sentences from the original article based on their relevance to the top-ranked entities, as described in the system manuscript. This evaluation measures how well the generated summary preserves the named entities that human editors considered important enough to include in the article headline.

---

## Gold Standard: Headline Entities as Automated Proxy

No benchmark dataset currently exists that pairs full news articles with human-written extractive summaries for this evaluation setup. As a result, article **headlines are used as an automated proxy** for central entity identification.

This approach is based from the work of **Singh et al. (2021)**, who demonstrate that named entities are among the most informationally significant elements of news headlines. Their study shows that preserving named entities is a primary concern when generating headlines from full articles, which supports the assumption that entities appearing in a headline are representative of the central entities in the corresponding article body.

### Important Limitation: NER-Dependent Gold Standard

The gold entities are extracted from headlines using the **same Flair NER model** that the EntiLytics system uses internally. This means the gold standard is not independently verified. Any systematic errors or biases in the Flair model will affect both the gold set and the system predictions. The headline-entity approach is an automated proxy for this configuration, but its precision as a gold standard cannot be fully guaranteed.

---

## Why Lower Bound Score is used

The Entity Coverage F1 reported here should be interpreted as a **conservative lower-bound baseline**, not an exact measure of summarization quality. There are three systematic reasons this score underestimates real performance:

**1. Entity density mismatch.** Headlines are maximally compressed and typically contain only 1 to 3 key entities. The extractive summary correctly includes additional salient entities that the headline had no room for. These additional correct entities in the summary reduce Precision without representing evaluation errors.

**2. Strict token matching penalty.** The evaluation uses exact token-level matching with no coreference resolution. A summary that writes "Department of Justice" when the headline says "DOJ" is counted as a miss, even though both refer to the same entity. Applying even partial fuzzy matching would produce a higher F1.

**3. Headline framing bias.** Headlines prioritize engagement hooks, while the extractive summary prioritizes entity coverage. A headline may name a person for rhetorical effect while the article body is primarily about an organization. The summary correctly emphasizes the organization, but the headline-based gold standard does not reward this.

---

## Evaluation Metric

**Entity Coverage F1** is used as the evaluation metric, consistent with Hofmann-Coyle et al. (2022), who establish F1 as the appropriate measure for entity-centric extractive summarization evaluated at the sentence level.

$$\text{Precision} = \frac{|\text{gold entities found in summary}|}{|\text{all ranked entities appearing in summary}|}$$

$$\text{Recall} = \frac{|\text{gold entities found in summary}|}{|\text{total gold entities}|}$$

$$F1 = \frac{2 \times \text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

---

## References

- Hofmann-Coyle, E., Kulkarni, M., Xie, L., Maddela, M., and Preotiuc-Pietro, D. (2022). Extractive entity-centric summarization as sentence selection using bi-encoders. In *Proceedings of the 2nd Conference of the Asia-Pacific Chapter of the Association for Computational Linguistics and the 12th International Joint Conference on Natural Language Processing* (Vol. 2, pp. 326-333).
- Singh, B., Marathe, A., Rizvi, A. A., and Joshi, A. R. (2021). Retaining named entities for headline generation. In *Inventive Computation and Information Technologies*.

---

## Section 1: Article Collection

Articles are fetched from three RSS feeds: **Manila Times**, **Tech Pinas**, and **The Guardian**. All articles available from each feed are collected with no per-feed cap. Each article must pass three filters:

- Non-empty headline and body text
- Body text of at least 100 characters (excludes extremely short RSS descriptions)
- No pipe character `|` in the headline (a common sign of templated or aggregated titles)

The collected articles are saved to `summarization_dataset.csv` for use in the evaluation section.

In [ ]:
import pandas as pd
import sys
from pathlib import Path
from bs4 import BeautifulSoup

base_path = Path(".").resolve()
sys.path.append(str(base_path.parent))

from features.rss_handler import fetch_rss_articles

# RSS feed sources
FEEDS = {
    "manila_times" : "https://www.manilatimes.net/news/feed/",
    "tech_pinas"   : "http://feeds.feedburner.com/Techpinas",
    "the_guardian" : "https://www.theguardian.com/international/rss",
}


def clean(text):
    """Remove HTML tags and normalize whitespace using BeautifulSoup."""
    if not text:
        return ""
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text(separator=" ").strip()


# Collect articles from all feeds
collected_article_rows = []
current_article_id = 1

for source_name, feed_url in FEEDS.items():
    raw_articles_list = fetch_rss_articles(feed_url)
    articles_collected_from_source = 0

    for article_data in raw_articles_list:
        article_headline  = clean(article_data.get("title", ""))
        article_full_text = clean(article_data.get("description", ""))

        # Skip invalid articles
        if not article_headline or not article_full_text:
            continue
        if len(article_full_text) < 100:
            continue
        if "|" in article_headline:
            continue

        collected_article_rows.append({
            "article_id" : current_article_id,
            "source"     : source_name,
            "headline"   : article_headline,
            "full_text"  : article_full_text,
        })
        current_article_id += 1
        articles_collected_from_source += 1

    print(f"{source_name}: collected {articles_collected_from_source} articles")

articles_dataframe = pd.DataFrame(collected_article_rows)
csv_output_path = base_path / "summarization_dataset.csv"
articles_dataframe.to_csv(csv_output_path, index=False)

print(f"\nTotal articles saved : {len(articles_dataframe)}")
print(f"Saved to             : {csv_output_path}")

---

## Section 2: Load and Filter Dataset

The saved CSV is loaded and stub articles are removed. Stub articles are RSS entries where the `full_text` field contains only a placeholder phrase such as "The post X appeared first on Y." with no actual article body. These cannot be meaningfully summarized and are excluded before evaluation.

In [ ]:
from features.flair_ner import identify_entities
from features.entity_ranking_and_summarization import entity_ranking, generate_summary

csv_dataset_path = base_path / "summarization_dataset.csv"
if not csv_dataset_path.exists():
    raise FileNotFoundError("summarization_dataset.csv not found. Run Section 1 first.")

df = pd.read_csv(csv_dataset_path)
print(f"Loaded {len(df)} articles")

# Remove RSS stub articles with no real body text
initial_count = len(df)
df = df[~df["full_text"].str.strip().str.match(r"^The post .+ appeared first on .+\.$")]
print(f"Removed {initial_count - len(df)} stub articles, {len(df)} remaining")

# Show article count per source
print()
print(df.groupby("source").size().rename("article count").to_frame().to_string())

---

## Section 3: Gold Standard Extraction

Named entities are extracted from each article headline using the Flair NER model. These headline entities serve as the gold standard for the evaluation.

In [ ]:
def extract_gold_entities_from_headline(headline_text):
    """Extract named entities from headline as gold standard."""
    extracted_entities = identify_entities(headline_text)
    return set(
        entity["text"].lower() for entity in extracted_entities
    )

---

## Section 4: Summarization Evaluation

For each article, the following steps are performed:

1. Gold entities are extracted from the headline using the function defined above.
2. Named entities are identified in the full article body using Flair NER.
3. `entity_ranking()` ranks those entities by importance based on position, frequency, and contextual relevance.
4. `generate_summary()` produces an extractive summary using the top-K ranked entities, where K equals the number of gold headline entities.
5. Entity Coverage F1 is computed by checking which gold entities appear anywhere in the generated summary text.

Articles where the headline yields no named entities are skipped and counted separately.

In [ ]:
# Evaluation loop
summary_precision_list = []
summary_recall_list = []
summary_f1_list = []
skipped_article_count = 0

print("SUMMARIZATION EVALUATION -- ENTITY COVERAGE F1\n")

for _, article_row in articles_dataframe.iterrows():
    # Extract gold entities from headline
    gold_headline_entities = extract_gold_entities_from_headline(article_row["headline"])

    if not gold_headline_entities:
        skipped_article_count += 1
        continue

    # Set top-K equal to number of gold entities
    article_top_k = max(1, len(gold_headline_entities))

    # Run summarization pipeline: extract entities -> rank -> generate summary
    body_flair_entities = identify_entities(article_row["full_text"])
    ranked_entities_list = entity_ranking(article_row["full_text"], body_flair_entities)
    generated_summary = generate_summary(article_row["full_text"], ranked_entities_list[:article_top_k])

    summary_text_lowercase = generated_summary["summary"].lower()

    # Find gold entities that appear in the summary
    gold_entities_in_summary = set(
        entity for entity in gold_headline_entities
        if any(word_token in summary_text_lowercase for word_token in entity.split())
    )

    # Find all ranked entities that appear in the summary
    all_ranked_entities_lowercase = set(
        entity["name"].lower() for entity in ranked_entities_list
    )
    all_ranked_entities_in_summary = set(
        entity for entity in all_ranked_entities_lowercase
        if any(word_token in summary_text_lowercase for word_token in entity.split())
    )

    # Calculate entity coverage metrics
    entity_true_positives = len(gold_entities_in_summary)
    
    entity_precision = (
        entity_true_positives / len(all_ranked_entities_in_summary)
        if all_ranked_entities_in_summary else 0
    )
    entity_recall = (
        entity_true_positives / len(gold_headline_entities)
        if gold_headline_entities else 0
    )
    entity_f1 = (
        (2 * entity_precision * entity_recall / (entity_precision + entity_recall))
        if (entity_precision + entity_recall) > 0 else 0
    )

    summary_precision_list.append(entity_precision)
    summary_recall_list.append(entity_recall)
    summary_f1_list.append(entity_f1)

    # Print per-article results
    print(f"[{article_row['source']}] {article_row['headline'][:65]}...")
    print(f"  Gold entities     : {gold_headline_entities}")
    print(f"  Found in summary  : {gold_entities_in_summary}")
    print(f"  P={entity_precision:.2f}  R={entity_recall:.2f}  F1={entity_f1:.2f}")
    print(f"  Summary ({generated_summary['sentence_count']} sentences): "
          f"{generated_summary['summary'][:120]}...\n")

---

## Section 5: Results

Mean Precision, Recall, and F1 are computed across all evaluated articles, excluding skipped ones. Results are reported as a **conservative lower-bound baseline** for the reasons described in the introduction.

In [ ]:
# Print aggregate results
total_evaluated_articles = len(summary_f1_list)
if total_evaluated_articles == 0:
    print("No articles evaluated.")
    sys.exit(0)

average_precision = sum(summary_precision_list) / total_evaluated_articles
average_recall = sum(summary_recall_list) / total_evaluated_articles
average_f1 = sum(summary_f1_list) / total_evaluated_articles

print(f"RESULTS ({total_evaluated_articles} articles evaluated, "
      f"{skipped_article_count} skipped)\n")
print("SUMMARIZATION -- Entity Coverage F1")
print(f"  Mean Precision : {average_precision:.4f}")
print(f"  Mean Recall    : {average_recall:.4f}")
print(f"  Mean F1        : {average_f1:.4f}")

---

## Section 6: Recorded Results and Interpretation

The results below were recorded from the evaluation run on **March 25, 2026**, using articles collected from Manila Times, Tech Pinas, and The Guardian.

| Metric | Value |
|---|---|
| Articles evaluated | 124 |
| Articles skipped (no headline entities) | 53 |
| Mean Precision | 0.6639 |
| Mean Recall | 0.8582 |
| Mean F1 | 0.6610 |

### Interpretation

**Recall (0.8582)** is high, meaning that in most articles the generated summary contains the key entities identified in the headline. The extractive approach consistently selects sentences that cover the main subjects of the article.

**Precision (0.6639)** is lower than Recall, which is expected behavior and not a system failure. The summary includes additional entities beyond those in the headline. These are entities that are genuinely salient in the article body but were omitted from the headline due to space constraints. Because the gold standard is limited to headline entities, these additional correct entities in the summary are not rewarded, which pulls Precision down.

**F1 (0.6610)** is the harmonic mean of Precision and Recall. As a lower-bound baseline, the true F1 under a human-annotated gold standard would be expected to be higher.

### Sample Outputs

The examples below illustrate the most common outcome patterns observed across the three news sources.

**Manila Times - Full coverage with precision penalty:**
```
[manila_times] Japan set to send combat troops to Philippine exercises - Brawner...
  Gold entities    : {'brawner', 'japan', 'philippine'}
  Found in summary : {'brawner', 'japan', 'philippine'}
  P=0.50  R=1.00  F1=0.67
  Summary (6 sentences): MANILA, Philippines - Japan is expected to send
  troops to military exercises in the Philippines...
```
All three gold entities were found in the summary. Precision is 0.50 because the summary also includes additional entities not present in the headline.

**The Guardian - Single entity, many additional correct entities in summary:**
```
[the_guardian] iPhone 17e review: Apple upgrades its cheapest new smartphone...
  Gold entities    : {'apple'}
  Found in summary : {'apple'}
  P=0.20  R=1.00  F1=0.33
  Summary (4 sentences): Mid-range handset gets chip, storage and MagSafe
  upgrades to offer more essential iOS features for less...
```
The summary correctly covers Apple but also includes iPhone, iOS, and MagSafe. These are genuinely relevant entities but absent from the headline gold set, reducing Precision to 0.20.

**Tech Pinas - Multi-entity coverage:**
```
[tech_pinas] Infinix Celebrates Aurora Gaming PH M7 Victory with Infinix GT 30...
  Gold entities    : {'aurora gaming', 'infinix'}
  Found in summary : {'infinix', 'aurora gaming'}
  P=0.22  R=1.00  F1=0.36
  Summary (24 sentences): Aurora Gaming PH celebrates its M7 World
  Championship triumph alongside Infinix...
```
Both gold entities appear in the summary. The lower Precision reflects the longer summary length for this article, which naturally contains more entity mentions from a longer source text.